# Model 1 — Resume Category Classifier
**PathCompanion AI · Phase 3 (CIS 6035)**

Classifies a resume into one of ~24 job categories. Two approaches are built and **compared**:
1. **Baseline** — TF-IDF + Logistic Regression
2. **Transformer** — fine-tuned DistilBERT

**Targets:** test accuracy ≥ 85%, macro-F1 ≥ 0.80.

**How to run**
1. `Runtime → Change runtime type → T4 GPU`.
2. Run each cell top-to-bottom.
3. Upload **`Resume.csv`** (Kaggle: `snehaanbhawal/resume-dataset`) when prompted.

In [ ]:
!pip -q install transformers datasets scikit-learn pandas matplotlib seaborn accelerate

## 1. Load the dataset
Upload **Resume.csv**. It has columns `Resume_str` (text) and `Category` (label).

In [ ]:
import pandas as pd
try:
    df = pd.read_csv('Resume.csv')
except FileNotFoundError:
    from google.colab import files
    up = files.upload()
    df = pd.read_csv(list(up.keys())[0])
print('rows, cols:', df.shape)
df.head(3)

In [ ]:
text_col = 'Resume_str' if 'Resume_str' in df.columns else ('Resume' if 'Resume' in df.columns else df.columns[-2])
df = df[[text_col, 'Category']].dropna().rename(columns={text_col: 'text', 'Category': 'label'})
print('categories:', df['label'].nunique(), '| rows:', len(df))
df['label'].value_counts()

## 2. Explore — category distribution

In [ ]:
import matplotlib.pyplot as plt
df['label'].value_counts().plot(kind='barh', figsize=(7, 7))
plt.title('Resumes per category'); plt.xlabel('count'); plt.tight_layout(); plt.show()

## 3. Clean text + train/test split (stratified 90/10)

In [ ]:
import re
from sklearn.model_selection import train_test_split

def clean(t):
    t = re.sub(r'http\S+', ' ', str(t))
    t = re.sub(r'[^a-zA-Z ]', ' ', t)
    return re.sub(r'\s+', ' ', t).lower().strip()

df['clean'] = df['text'].apply(clean)
train_df, test_df = train_test_split(df, test_size=0.1, stratify=df['label'], random_state=42)
print('train:', len(train_df), ' test:', len(test_df))

## 4. Baseline — TF-IDF + Logistic Regression

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, f1_score, classification_report

baseline = Pipeline([
    ('tfidf', TfidfVectorizer(stop_words='english', max_features=20000, ngram_range=(1, 2))),
    ('clf', LogisticRegression(max_iter=2000)),
])
baseline.fit(train_df['clean'], train_df['label'])
pred = baseline.predict(test_df['clean'])
base_acc = accuracy_score(test_df['label'], pred)
base_f1 = f1_score(test_df['label'], pred, average='macro')
print(f'Baseline  Accuracy={base_acc:.3f}  Macro-F1={base_f1:.3f}')
print(classification_report(test_df['label'], pred))

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay
fig, ax = plt.subplots(figsize=(10, 10))
ConfusionMatrixDisplay.from_estimator(baseline, test_df['clean'], test_df['label'],
                                      xticks_rotation='vertical', ax=ax, colorbar=False)
plt.title('Baseline — confusion matrix'); plt.tight_layout(); plt.show()

## 5. Transformer — fine-tune DistilBERT
Needs the **T4 GPU** runtime. Takes ~5–10 minutes.

In [ ]:
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df['y'] = le.fit_transform(df['label'])
num_labels = len(le.classes_)
tr = df.loc[train_df.index]
te = df.loc[test_df.index]
print('num_labels:', num_labels)

In [ ]:
from datasets import Dataset
from transformers import AutoTokenizer

ckpt = 'distilbert-base-uncased'
tok = AutoTokenizer.from_pretrained(ckpt)

def tokenize(b):
    return tok(b['text'], truncation=True, padding='max_length', max_length=256)

tr_ds = Dataset.from_pandas(tr[['text', 'y']].rename(columns={'y': 'labels'}), preserve_index=False).map(tokenize, batched=True)
te_ds = Dataset.from_pandas(te[['text', 'y']].rename(columns={'y': 'labels'}), preserve_index=False).map(tokenize, batched=True)

In [ ]:
import numpy as np, torch
from transformers import AutoModelForSequenceClassification, TrainingArguments, Trainer
from sklearn.metrics import accuracy_score, f1_score

model = AutoModelForSequenceClassification.from_pretrained(ckpt, num_labels=num_labels)

def compute_metrics(p):
    preds = np.argmax(p.predictions, axis=1)
    return {'accuracy': accuracy_score(p.label_ids, preds),
            'macro_f1': f1_score(p.label_ids, preds, average='macro')}

args = TrainingArguments(
    output_dir='out', num_train_epochs=4,
    per_device_train_batch_size=16, per_device_eval_batch_size=32,
    learning_rate=2e-5, logging_steps=20, report_to='none',
    fp16=torch.cuda.is_available(),
)
trainer = Trainer(model=model, args=args, train_dataset=tr_ds, eval_dataset=te_ds, compute_metrics=compute_metrics)
trainer.train()

In [ ]:
res = trainer.evaluate()
bert_acc = res['eval_accuracy']; bert_f1 = res['eval_macro_f1']
print(f"DistilBERT  Accuracy={bert_acc:.3f}  Macro-F1={bert_f1:.3f}")

## 6. Compare the two models
For the dissertation, also run a **zero-shot LLM baseline** (ask Gemini to classify ~50 test resumes) and add its accuracy to this table.

In [ ]:
import pandas as pd
print(pd.DataFrame({
    'Model': ['TF-IDF + LogReg', 'DistilBERT'],
    'Accuracy': [round(base_acc, 3), round(bert_acc, 3)],
    'Macro-F1': [round(base_f1, 3), round(bert_f1, 3)],
}).to_string(index=False))

## 7. Save models (download for the backend, Phase 5)

In [ ]:
import joblib, json, shutil
joblib.dump(baseline, 'model1_resume_classifier_baseline.joblib')
json.dump(list(le.classes_), open('model1_labels.json', 'w'))
model.save_pretrained('model1_distilbert'); tok.save_pretrained('model1_distilbert')
shutil.make_archive('model1_distilbert', 'zip', 'model1_distilbert')

from google.colab import files
files.download('model1_resume_classifier_baseline.joblib')
files.download('model1_labels.json')
files.download('model1_distilbert.zip')
print('Saved + downloading. Keep these for backend integration.')